In [1]:
import numpy as np
import os
import time
from params import gen_amplitude_params, gen_sky_location_params, gen_frequency_params
from params import gen_glitch_params, save_params
import yaml 

with open('/scratch/kriles_root/kriles0/damoncht/cwglitch/config/config.yaml', 'r') as f:
    config = yaml.safe_load(f)

with open('/scratch/kriles_root/kriles0/damoncht/cwglitch/targets/casa.yaml', 'r') as f:
    target = yaml.safe_load(f)


In [2]:
fmin, fmax = 100, 100

label = 'wg_dnu_nu_1e-6_gaussian_gap'
out_dir = f'/scratch/kriles_root/kriles0/damoncht/cwglitch/data/{label}/{fmin}-{fmax}Hz'

# Define your parameters
tau = target['age'] * 365.25 * 86400
freq = 100
f1min, f1max = -freq/tau, -freq/tau
f2min, f2max = 0, 0 

params = {
    'n': 8,
    'm': 1,
    'age': tau,
    'freq_ranges': [(fmin, fmax), (f1min, f1max), (f2min, f2max)],
    'freq_order': 2,
    'glitch_params_ranges': {
        'tglitch': (0, 1), 
        'dnu_nu': (1e-6, 1e-6),          # Updated key
        'dnu1_nu1': (0, 0),              # Updated key
        'Q': (0, 0),   
        'tau': (10*86400, 10*86400)
    },
    'alpha': target['alpha'],
    'delta': target['delta'],
    'seed': 0, 
}

n, m = params['n'], params['m']
age = params['age']
freq_ranges, freq_order = params['freq_ranges'], params['freq_order']
glitch_params_ranges = params['glitch_params_ranges']

if params.get('seed') is not None:
    np.random.seed(params['seed'])
    print(f"Using random seed: {params['seed']}")

freq_params = gen_frequency_params(n, freq_order, freq_ranges)
amp_params = gen_amplitude_params(n)

if params.get('alpha') is not None and params.get('delta') is not None:
    sky_params = np.column_stack((np.full(n, params['alpha']), np.full(n, params['delta'])))
else:
    print("Generating sky location.")
    sky_params = gen_sky_location_params(n)
    
glitch_params = gen_glitch_params(
    n=n, 
    m=m, 
    tglitch_range=glitch_params_ranges['tglitch'],
    dnu_nu_range=glitch_params_ranges['dnu_nu'],
    dnu1_nu1_range=glitch_params_ranges['dnu1_nu1'],
    Q_range=glitch_params_ranges['Q'],
    tau_range=glitch_params_ranges['tau']
)
        
if m > 0:
    tglitch_spaced = np.linspace(glitch_params_ranges['tglitch'][0], glitch_params_ranges['tglitch'][1], n)
    for i in range(n):
        glitch_params[i][0][0] = tglitch_spaced[i]

freq_params_padded = np.zeros((n, 5))
freq_params_padded[:, :freq_order+1] = freq_params

fmin, fmax = freq_ranges[0]
os.makedirs(out_dir, exist_ok=True)

# 4. Updated save_params call (Passing the filepath directly)
csv_filepath = os.path.join(out_dir, 'signal_glitch_params.csv')
save_params(
    n, m,
    freq_params_padded, amp_params, sky_params, glitch_params, 
    filepath=csv_filepath
)

Using random seed: 0
Generating glitch parameters.
